# 02. Intermediate models
**Case study:** Partisanship & economic perception · **Layer:** intermediate (staging tables → analysis-ready building blocks)

## What an intermediate model does
Staging gave one clean table per raw source. The intermediate layer is where **analysis decisions** live:
* **Harmonize** sources that measure the same thing differently (CES and ANES party, outcomes, controls)
* **Join** across sources (economic series onto one monthly calendar; sentiment next to actual inflation)
* **Derive** analysis variables (better/worse binaries, in-party vs out-party, year-over-year changes, event time)
* **Apply** the analysis window (2016 onward for the surveys)

Every coding choice the plan's *Analytical design* section describes is made here, once, so a reviewer can audit
it in one notebook. Marts (next notebook) only aggregate these tables; they never recode.

## Models built here
| Model | Built from | Grain | Feeds |
|---|---|---|---|
| `ref_midterms` | hand-built seed | midterm election | midterm placebo (RQ1 validity check) |
| `int_presidential_terms` | `ref_events` | presidential term | "who held the White House on date X" for every model below |
| `int_respondents` | `stg_ces`, `stg_anes` | respondent | RQ1 year-FE model, RQ3 wallet-vs-economy gaps |
| `int_econ_monthly` | `stg_inflation`, `stg_cpi`, `stg_misery_index`, `stg_fred_observations` | month (2000 → 2026) | fundamentals everywhere |
| `int_sentiment_monthly` | `stg_umich_party`, `stg_umich_index`, `int_econ_monthly` | month (2016 → 2026) | RQ1 monthly series, RQ2 expected vs actual inflation |
| `int_econ_windows` | `int_respondents`, `int_econ_monthly` | survey × year × window length | RQ2: fundamentals in the months before each survey |
| `int_event_panel` | `ref_events`, `ref_midterms`, `int_sentiment_monthly`, `int_econ_monthly` | event × month × party | RQ1 event study, fundamentals-at-transitions table, midterm placebo |
| `int_pew_polls` | `stg_pew`, `ref_events` | poll × party | RQ1 Pew cross-check (last poll before vs first after) |

## Key decisions made in this notebook (each is explained where it happens)
1. **"Not sure" answers.** The main-spec binaries (`econ_better`, `econ_worse`) set "not sure" to NULL, i.e. **excluded**, as
   the plan's main spec says. A second set (`*_ns_as_same`) folds "not sure" into "stayed the same", which is the plan's A7
   robustness check and is also exactly what the preliminary numbers did. Both are kept so neither is a hidden choice.
2. **In-party = party holding the White House on the fieldwork date**, looked up from `int_presidential_terms`. ANES has no
   interview date, so ANES uses an assumed reference date of 1 October of the survey year (flagged per row).
3. **Gap-free monthly calendar.** Every monthly model is built on a complete month spine, so `LAG(x, 12)` always means
   "12 months earlier" and a missing month (October 2025) shows up as a NULL row rather than silently shifting a lag.
4. **No filling.** October 2025 (no CPI, no unemployment published) stays NULL everywhere; windows that touch it report
   how many months they actually averaged.
5. **Event-study baseline = the three months ending two months before the election** (July–September for a November
   election). That is what the plan's preliminary table used; it keeps October's late-campaign movement out of the baseline.

## Conventions (same as notebook 01)
* Run top to bottom; every cell is `CREATE OR REPLACE`, so re-running is safe. Run notebook 01 first.
* Tables land in `workspace.public_opinion_economy_case_study` with an `int_` prefix (seeds use `ref_`).
* **dbt mapping:** each `SELECT` is the body of `dbt/models/intermediate/<name>.sql`; swap table names for `{{ ref('...') }}`.
  `ref_midterms` becomes a CSV in `dbt/seeds/`.

# 0. Setup

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA public_opinion_economy_case_study;

-- Same as notebook 01: date extraction from timestamps is done in UTC so results never depend on the cluster's time zone.
SET TIME ZONE 'UTC';

## 1. `ref_midterms` — midterm elections (seed)
**Grain:** one row per midterm election, 2006–2022 (5 rows).

The midterm placebo asks: *if perceptions follow power, do they move when Congress changes hands but the White House
does not?* Every midterm since 2006 flipped at least one chamber, so each is a usable placebo. In all five the party
**out** of the White House won, which is worth saying in the write-up (the placebo always tests the out-party gaining power).

| Year | Chamber(s) flipped | To |
|---|---|---|
| 2006 | House and Senate | Democrats |
| 2010 | House | Republicans |
| 2014 | Senate | Republicans |
| 2018 | House | Democrats |
| 2022 | House | Republicans |

Only 2018 and 2022 fall inside the UMich party series (Apr 2017 onward) and the CES odd-year comparison (2018→2019,
2022→2023); the earlier rows are there so the seed is complete if the series are extended back.
The upcoming 3 Nov 2026 midterm is added as a row once results are certified.

In [0]:
%sql
CREATE OR REPLACE TABLE ref_midterms
COMMENT 'Hand-built seed: US midterm elections 2006-2022 with the party that gained control of a chamber. Used as placebo events (Congress changes hands, White House does not).'
AS
SELECT
  election_year,
  CAST(election_date AS DATE)          AS election_date,
  trunc(CAST(election_date AS DATE), 'MM') AS election_month,   -- join key for monthly series
  winning_party,                                                -- party that took control of the flipped chamber(s)
  chambers_flipped
FROM VALUES
  (2006, '2006-11-07', 'Democrat',   'House and Senate'),
  (2010, '2010-11-02', 'Republican', 'House'),
  (2014, '2014-11-04', 'Republican', 'Senate'),
  (2018, '2018-11-06', 'Democrat',   'House'),
  (2022, '2022-11-08', 'Republican', 'House')
AS t(election_year, election_date, winning_party, chambers_flipped);

## 2. `int_presidential_terms` — who held the White House, and when
**Grain:** one row per four-year presidential term (8 rows: Clinton's second term through the current term).

`ref_events` is organized around *transitions* (election → inauguration). Most models need the opposite question:
*"which party held the White House on this date?"* That is a range lookup, so this model turns transitions into terms with a
start and end date. Every downstream model then joins with `date BETWEEN term_start_date AND term_end_date`.

* **Clinton's second term (1997–2001) is added by hand** because the events file starts with the 2000 election, and the monthly
  inflation series starts in January 2000. Without it, 2000 would have no White House party.
* A president's two terms are two rows (e.g., Bush 2001 and Bush 2005), which keeps re-elections visible as their own events.
* `lame_duck_start_date` is the election that the incumbent's party **lost** (NULL if the successor is the same party). Between
  that date and the next inauguration, the outgoing party still holds office but the result is known; the plan's revised
  timing hypothesis is that sentiment starts moving here.
* The current term ends on `9999-12-31` so open-ended range joins work; `is_current_term` flags it.
* Inauguration day itself is assigned to the incoming president (the oath is at noon; date-level data cannot split the day).

In [0]:
%sql
CREATE OR REPLACE TABLE int_presidential_terms
COMMENT 'One row per presidential term (1997 onward) with start/end dates for range joins, party, predecessor and successor party, and the lame-duck start date when the White House changed parties.'
AS
WITH transitions AS (
  SELECT election_date, inauguration_date, president_name, incoming_party
  FROM ref_events

  UNION ALL

  -- Hand-added: Clinton's second term, so dates in 1997-2000 resolve to a party.
  SELECT DATE'1996-11-05', DATE'1997-01-20', 'Bill Clinton', 'Democrat'
),

ordered AS (
  SELECT
    *,
    LAG(incoming_party)     OVER (ORDER BY inauguration_date) AS predecessor_party,
    LEAD(inauguration_date) OVER (ORDER BY inauguration_date) AS next_inauguration_date,
    LEAD(election_date)     OVER (ORDER BY inauguration_date) AS next_election_date,
    LEAD(incoming_party)    OVER (ORDER BY inauguration_date) AS successor_party,
    LEAD(president_name)    OVER (ORDER BY inauguration_date) AS successor_name
  FROM transitions
)

SELECT
  president_name,
  incoming_party                                                AS party,
  election_date                                                 AS won_election_date,
  inauguration_date                                             AS term_start_date,
  -- Term ends the day before the next inauguration; the current term is open-ended.
  COALESCE(date_sub(next_inauguration_date, 1), DATE'9999-12-31') AS term_end_date,
  next_inauguration_date IS NULL                                AS is_current_term,
  predecessor_party,                                            -- NULL for the first row (Clinton 1997)
  incoming_party <> predecessor_party                           AS took_over_from_other_party,
  successor_name,
  successor_party,
  next_election_date                                            AS successor_election_date,
  -- Only when the White House is about to change parties: from the election that the incumbent party lost.
  CASE WHEN successor_party <> incoming_party THEN next_election_date END AS lame_duck_start_date
FROM ordered;

## 3. `int_respondents` — CES and ANES on one harmonized respondent table
**Grain:** one row per respondent. **Window:** survey years 2016 onward (CES 2016–2025, ANES 2016 / 2020 / 2024).
The window is the one line in the final `WHERE`; widen it there to bring in the 2008–09 transition later.

### Why stack the two surveys
RQ1 is estimated on CES; RQ3 on ANES. Both need the same party groups, the same outcome coding and the same in-party flag.
Coding them side by side in one model guarantees that "Republican", "better" and "in-party" mean the same thing in every
chart, which is the plan's harmonization promise. Analyses filter on `source`; **never pool weights across sources.**

### Outcome coding (plan: *Outcome coding, shared across RQs*)
| Column | Better | Same | Worse | Not sure | Use |
|---|---|---|---|---|---|
| `econ_retro3` | `'Better'` | `'Same'` | `'Worse'` | `'Not sure'` | descriptive tables |
| `econ_better` / `econ_worse` | 1 / 0 | 0 / 0 | 0 / 1 | **NULL** | **main spec** (not sure excluded) |
| `econ_better_ns_as_same` / `econ_worse_ns_as_same` | 1 / 0 | 0 / 0 | 0 / 1 | 0 / 0 | robustness A7; matches the preliminary numbers |
| `econ_net` | +1 | 0 | −1 | NULL | weighted mean = net rating (% better − % worse) |
| `econ_score5` | 4–5 | 3 | 1–2 | NULL | robustness: 5-point mean / ordered logit (5 = much better) |

The same set exists for **own finances** (`personal_*`), which only ANES asks. CES rows carry NULLs there.
A weighted mean of any 0/1 column is directly a weighted share, e.g. `SUM(econ_better * weight) / SUM(weight)` over rows where
`econ_better IS NOT NULL`.

### Party and in-party coding
* `party3` keeps leaners with their party (plan's convention in every source). `party3_strict` moves leaners to Independent
  (robustness battery: *leaners with parties vs leaners as independents*).
* `white_house_party` = party in office on `fieldwork_ref_date`. `partisan_role` = In-party / Out-party / Independent.
* `co_partisan` is the plan's β3 regressor: 1 if the respondent's party holds the White House, 0 for the other party **and for
  independents**, NULL for "not sure". (`*_strict` versions follow `party3_strict`.)

### Controls harmonized across sources (the plan's X in the RQ1 model, and RQ3 controls)
| Column | CES source | ANES source | Note |
|---|---|---|---|
| `age_group` | `age` | `age` | 18–34 / 35–54 / 55+, matching the UMich age subgroups |
| `educ3` | `educ` (6 levels) | `VCF0110` (4 levels) | High school or less / Some college (incl. 2-year degree) / College degree or more, matching UMich |
| `race4` | `race_h` | `VCF0105b` | White / Black / Hispanic / Other. CES Asian, Native American, Mixed, Middle Eastern → Other because ANES only has 4 groups |
| `income_tercile` | `faminc` brackets | `VCF0114` percentile bands | ANES bands map exactly. **CES is approximate**: each dollar bracket is placed by the weighted percentile at its midpoint within the year (see the CTE). "Prefer not to say" kept as its own category |
| `employment5` | `employ` | `VCF0118` | CES is collapsed into ANES's five groups. "Permanently disabled", "temporarily laid off" and "other" go to *Not employed*, as in ANES |
| `owns_home`, `is_stock_investor` | `ownhome`, `investor` | `VCF0146`, `VCF9224` | CES "Other" housing → not an owner. CES `investor` not asked in 2023 or 2025 |
| `state_abbr` | `st` | `VCF0901a` FIPS → postal code | lookup below; one consistent key for state fixed effects |

In [0]:
%sql
CREATE OR REPLACE TABLE int_respondents
COMMENT 'CES 2016-2025 and ANES 2016/2020/2024 respondents on one harmonized schema: party (with and without leaners), White House party at fieldwork, in-party flags, better/worse/net outcomes for the national economy (both) and own finances (ANES), harmonized controls.'
AS
WITH
-- ─────────────────────────────────────────────────────────────────────────────
-- Lookup: state FIPS code -> postal abbreviation (ANES stores FIPS; CES stores postal codes).
-- ─────────────────────────────────────────────────────────────────────────────
state_fips AS (
  SELECT * FROM VALUES
    ('01','AL'),('02','AK'),('04','AZ'),('05','AR'),('06','CA'),('08','CO'),('09','CT'),('10','DE'),('11','DC'),
    ('12','FL'),('13','GA'),('15','HI'),('16','ID'),('17','IL'),('18','IN'),('19','IA'),('20','KS'),('21','KY'),
    ('22','LA'),('23','ME'),('24','MD'),('25','MA'),('26','MI'),('27','MN'),('28','MS'),('29','MO'),('30','MT'),
    ('31','NE'),('32','NV'),('33','NH'),('34','NJ'),('35','NM'),('36','NY'),('37','NC'),('38','ND'),('39','OH'),
    ('40','OK'),('41','OR'),('42','PA'),('44','RI'),('45','SC'),('46','SD'),('47','TN'),('48','TX'),('49','UT'),
    ('50','VT'),('51','VA'),('53','WA'),('54','WV'),('55','WI'),('56','WY')
  AS t(state_fips, state_abbr)
),

-- ─────────────────────────────────────────────────────────────────────────────
-- CES income terciles (approximate).
-- CES asks income in dollar brackets, and the same bracket is a different place in the distribution each year.
-- For each year: weighted share of respondents in each bracket -> cumulative share -> the bracket's MIDPOINT
-- percentile decides its tercile. Brackets are coarse, so realized tercile shares are roughly, not exactly, a third
-- (the QA cell at the end prints them). Respondents who preferred not to say are excluded here and labeled below.
-- ─────────────────────────────────────────────────────────────────────────────
ces_income_brackets AS (
  SELECT survey_year, family_income_rank, SUM(weight) AS bracket_weight
  FROM stg_ces
  WHERE survey_year >= 2016 AND family_income_rank IS NOT NULL
  GROUP BY survey_year, family_income_rank
),

ces_income_terciles AS (
  SELECT
    survey_year,
    family_income_rank,
    -- cumulative share up to and including this bracket, minus half of this bracket = midpoint percentile
    ( SUM(bracket_weight) OVER (PARTITION BY survey_year ORDER BY family_income_rank
                                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
      - bracket_weight / 2 )
    / SUM(bracket_weight) OVER (PARTITION BY survey_year)                 AS bracket_midpoint_pctile
  FROM ces_income_brackets
),

-- ─────────────────────────────────────────────────────────────────────────────
-- CES: map every column to the shared schema. Column ORDER must match the ANES block exactly (UNION ALL is positional).
-- ─────────────────────────────────────────────────────────────────────────────
ces AS (
  SELECT
    'CES'                                                       AS source,
    c.survey_year,
    c.respondent_key,
    c.weight,
    c.weight_cumulative,

    -- Fieldwork timing: CES has a real start date per respondent.
    c.start_date_utc                                            AS fieldwork_date,
    c.start_date_utc                                            AS fieldwork_ref_date,
    FALSE                                                       AS fieldwork_date_is_assumed,

    -- Party
    c.party3,                                                   -- Democrat / Republican / Independent / Not sure
    CASE
      WHEN c.pid7_code IN (1, 2)    THEN 'Democrat'
      WHEN c.pid7_code IN (6, 7)    THEN 'Republican'
      WHEN c.pid7_code IN (3, 4, 5) THEN 'Independent'          -- leaners (3, 5) moved to Independent
      WHEN c.pid7_code = 8          THEN 'Not sure'
    END                                                         AS party3_strict,
    c.pid7,

    -- National economy, past year (CES economy_retro: 1 much better ... 5 much worse, 6 not sure)
    CASE
      WHEN c.economy_retro_code IN (1, 2) THEN 'Better'
      WHEN c.economy_retro_code = 3       THEN 'Same'
      WHEN c.economy_retro_code IN (4, 5) THEN 'Worse'
      WHEN c.economy_retro_code = 6       THEN 'Not sure'
    END                                                         AS econ_retro3,
    CASE WHEN c.economy_retro_code BETWEEN 1 AND 5
         THEN 6 - c.economy_retro_code END                      AS econ_score5,      -- flip so 5 = much better

    -- Own finances: not asked in CES
    CAST(NULL AS STRING)                                        AS personal_retro3,
    CAST(NULL AS INT)                                           AS personal_score5,

    -- Controls
    c.age,
    c.sex,
    CASE
      WHEN c.educ_code IN (1, 2) THEN 'High school or less'
      WHEN c.educ_code IN (3, 4) THEN 'Some college'            -- includes 2-year degrees (ANES does the same)
      WHEN c.educ_code IN (5, 6) THEN 'College degree or more'
    END                                                         AS educ3,
    CASE
      WHEN c.race_ethnicity IN ('White', 'Black', 'Hispanic') THEN c.race_ethnicity
      WHEN c.race_ethnicity IS NOT NULL                       THEN 'Other'
    END                                                         AS race4,
    CASE
      WHEN c.family_income = 'Prefer not to say' THEN 'Prefer not to say'
      WHEN t.bracket_midpoint_pctile < 1.0 / 3   THEN 'Bottom third'
      WHEN t.bracket_midpoint_pctile < 2.0 / 3   THEN 'Middle third'
      WHEN t.bracket_midpoint_pctile IS NOT NULL THEN 'Top third'
    END                                                         AS income_tercile,
    CASE
      WHEN c.employment_status IN ('Full-Time', 'Part-Time')    THEN 'Employed'
      WHEN c.employment_status IN ('Unemployed', 'Temporarily Laid Off', 'Permanently Disabled', 'Other')
                                                                THEN 'Not employed'
      WHEN c.employment_status = 'Retired'                      THEN 'Retired'
      WHEN c.employment_status = 'Homemaker'                    THEN 'Homemaker'
      WHEN c.employment_status = 'Student'                      THEN 'Student'
    END                                                         AS employment5,
    CASE c.home_ownership WHEN 'Own' THEN TRUE WHEN 'Rent' THEN FALSE WHEN 'Other' THEN FALSE END AS owns_home,
    CASE c.stock_investor WHEN 'Yes' THEN TRUE WHEN 'No' THEN FALSE END                           AS is_stock_investor,

    -- Geography
    c.state_abbr,
    c.county_fips,                                              -- CES only: joins to BLS LAUS county unemployment
    c.zipcode,                                                  -- CES only: joins to Imprint ZIP-level aggregates

    -- Source-specific extras kept for robustness checks
    CAST(NULL AS STRING)                                        AS interview_mode       -- ANES only
  FROM stg_ces AS c
  LEFT JOIN ces_income_terciles AS t
    ON c.survey_year = t.survey_year AND c.family_income_rank = t.family_income_rank
  WHERE c.survey_year >= 2016
),

-- ─────────────────────────────────────────────────────────────────────────────
-- ANES: same schema, same order.
-- ─────────────────────────────────────────────────────────────────────────────
anes AS (
  SELECT
    'ANES'                                                      AS source,
    a.survey_year,
    a.respondent_key,
    a.weight,
    CAST(NULL AS DOUBLE)                                        AS weight_cumulative,  -- CES only

    -- ANES cumulative file has no interview date. ASSUMPTION: 1 October of the survey year, inside the fall
    -- pre-election fieldwork period, where both outcome items are asked. Flagged so it is never mistaken for a real date.
    CAST(NULL AS DATE)                                          AS fieldwork_date,
    make_date(a.survey_year, 10, 1)                             AS fieldwork_ref_date,
    TRUE                                                        AS fieldwork_date_is_assumed,

    a.party3,                                                   -- Democrat / Republican / Independent (leaners in parties)
    CASE
      WHEN a.pid7_code IN (1, 2)    THEN 'Democrat'
      WHEN a.pid7_code IN (6, 7)    THEN 'Republican'
      WHEN a.pid7_code IN (3, 4, 5) THEN 'Independent'
    END                                                         AS party3_strict,
    a.pid7,

    -- National economy, past year (VCF0870 labels from staging)
    CASE a.econ_national_retro
      WHEN 'Better'          THEN 'Better'
      WHEN 'Stayed the same' THEN 'Same'
      WHEN 'Worse'           THEN 'Worse'
      WHEN 'Do not know'     THEN 'Not sure'
    END                                                         AS econ_retro3,
    CASE a.econ_national_retro_5pt                              -- VCF0871, 5 = much better
      WHEN 'Much better' THEN 5 WHEN 'Somewhat better' THEN 4 WHEN 'Stayed the same' THEN 3
      WHEN 'Somewhat worse' THEN 2 WHEN 'Much worse' THEN 1
    END                                                         AS econ_score5,

    -- Own finances, past year (VCF0880 / VCF0880a)
    CASE a.personal_fin_retro
      WHEN 'Better'          THEN 'Better'
      WHEN 'Stayed the same' THEN 'Same'
      WHEN 'Worse'           THEN 'Worse'
      WHEN 'Do not know'     THEN 'Not sure'
    END                                                         AS personal_retro3,
    CASE a.personal_fin_retro_5pt
      WHEN 'Much better' THEN 5 WHEN 'Somewhat better' THEN 4 WHEN 'Stayed the same' THEN 3
      WHEN 'Somewhat worse' THEN 2 WHEN 'Much worse' THEN 1
    END                                                         AS personal_score5,

    a.age,
    CASE WHEN a.gender IN ('Male', 'Female') THEN a.gender END  AS sex,     -- 'Other' (2016 only, a handful) -> NULL to match CES
    CASE
      WHEN a.educ_code IN (1, 2) THEN 'High school or less'
      WHEN a.educ_code = 3       THEN 'Some college'
      WHEN a.educ_code = 4       THEN 'College degree or more'
    END                                                         AS educ3,
    CASE a.race_ethnicity
      WHEN 'White non-Hispanic'              THEN 'White'
      WHEN 'Black non-Hispanic'              THEN 'Black'
      WHEN 'Hispanic'                        THEN 'Hispanic'
      WHEN 'Other or multiple, non-Hispanic' THEN 'Other'
    END                                                         AS race4,
    CASE
      WHEN a.income_pctile_code IN (1, 2) THEN 'Bottom third'   -- 0-16th and 17-33rd percentiles
      WHEN a.income_pctile_code = 3       THEN 'Middle third'   -- 34-67th
      WHEN a.income_pctile_code IN (4, 5) THEN 'Top third'      -- 68-95th and 96-100th
    END                                                         AS income_tercile,
    CASE a.work_status
      WHEN 'Employed'  THEN 'Employed'
      WHEN 'Retired'   THEN 'Retired'
      WHEN 'Homemaker' THEN 'Homemaker'
      WHEN 'Student'   THEN 'Student'
      WHEN 'Not employed (laid off, unemployed, disabled, other)' THEN 'Not employed'
    END                                                         AS employment5,
    CASE a.home_ownership WHEN 'Own' THEN TRUE WHEN 'Do not own' THEN FALSE END AS owns_home,   -- 51% NULL in 2020
    CASE a.stock_investor WHEN 'Yes' THEN TRUE WHEN 'No' THEN FALSE END         AS is_stock_investor,

    s.state_abbr,                                               -- 44% NULL in 2024 (blank in source)
    CAST(NULL AS STRING)                                        AS county_fips,
    CAST(NULL AS STRING)                                        AS zipcode,

    a.interview_mode
  FROM stg_anes AS a
  LEFT JOIN state_fips AS s ON a.state_fips = s.state_fips
  WHERE a.survey_year >= 2016
),

stacked AS (
  SELECT * FROM ces
  UNION ALL
  SELECT * FROM anes
),

-- ─────────────────────────────────────────────────────────────────────────────
-- Attach the White House party on each respondent's reference date (range join to terms).
-- ─────────────────────────────────────────────────────────────────────────────
with_white_house AS (
  SELECT
    r.*,
    t.party                                                     AS white_house_party,
    t.president_name                                            AS president_at_fieldwork,
    -- TRUE for the first fall survey after the White House changed parties (CES 2017, 2021, 2025): the RQ1 contrast years.
    t.took_over_from_other_party AND year(t.term_start_date) = r.survey_year AS is_first_year_after_party_change
  FROM stacked AS r
  LEFT JOIN int_presidential_terms AS t
    ON r.fieldwork_ref_date BETWEEN t.term_start_date AND t.term_end_date
)

SELECT
  -- ── Keys, weights, timing ──────────────────────────────────────────────────
  source,
  survey_year,
  respondent_key,
  weight,                                       -- within-source, within-year weight (use for every estimate)
  weight_cumulative,                            -- CES only, pooled multi-year estimates
  fieldwork_date,
  fieldwork_ref_date,
  trunc(fieldwork_ref_date, 'MM')               AS fieldwork_month,
  fieldwork_date_is_assumed,
  survey_year IN (2020, 2021)                   AS is_covid_year,    -- robustness: "excluding 2020-2021 (COVID)"

  -- ── White House at fieldwork ───────────────────────────────────────────────
  white_house_party,
  president_at_fieldwork,
  is_first_year_after_party_change,

  -- ── Party (leaners with parties: main spec) ────────────────────────────────
  party3,
  CASE
    WHEN party3 IN ('Democrat', 'Republican') THEN IF(party3 = white_house_party, 'In-party', 'Out-party')
    WHEN party3 = 'Independent'               THEN 'Independent'
  END                                           AS partisan_role,     -- NULL for "Not sure" and missing party
  CASE
    WHEN party3 IN ('Democrat', 'Republican') THEN IF(party3 = white_house_party, 1, 0)
    WHEN party3 = 'Independent'               THEN 0
  END                                           AS co_partisan,       -- plan's beta-3 regressor

  -- ── Party (leaners as independents: robustness) ────────────────────────────
  party3_strict,
  CASE
    WHEN party3_strict IN ('Democrat', 'Republican') THEN IF(party3_strict = white_house_party, 'In-party', 'Out-party')
    WHEN party3_strict = 'Independent'               THEN 'Independent'
  END                                           AS partisan_role_strict,
  CASE
    WHEN party3_strict IN ('Democrat', 'Republican') THEN IF(party3_strict = white_house_party, 1, 0)
    WHEN party3_strict = 'Independent'               THEN 0
  END                                           AS co_partisan_strict,
  pid7,

  -- ── National economy, past year ────────────────────────────────────────────
  econ_retro3,
  CASE econ_retro3 WHEN 'Better' THEN 1 WHEN 'Same' THEN 0 WHEN 'Worse' THEN 0 END   AS econ_better,           -- main spec: not sure -> NULL
  CASE econ_retro3 WHEN 'Better' THEN 0 WHEN 'Same' THEN 0 WHEN 'Worse' THEN 1 END   AS econ_worse,
  CASE econ_retro3 WHEN 'Better' THEN 1 WHEN 'Same' THEN 0 WHEN 'Worse' THEN -1 END  AS econ_net,              -- mean = % better - % worse
  CASE WHEN econ_retro3 IS NOT NULL THEN IF(econ_retro3 = 'Better', 1, 0) END        AS econ_better_ns_as_same, -- robustness A7
  CASE WHEN econ_retro3 IS NOT NULL THEN IF(econ_retro3 = 'Worse',  1, 0) END        AS econ_worse_ns_as_same,
  econ_score5,

  -- ── Own finances, past year (ANES only) ────────────────────────────────────
  personal_retro3,
  CASE personal_retro3 WHEN 'Better' THEN 1 WHEN 'Same' THEN 0 WHEN 'Worse' THEN 0 END  AS personal_better,
  CASE personal_retro3 WHEN 'Better' THEN 0 WHEN 'Same' THEN 0 WHEN 'Worse' THEN 1 END  AS personal_worse,
  CASE personal_retro3 WHEN 'Better' THEN 1 WHEN 'Same' THEN 0 WHEN 'Worse' THEN -1 END AS personal_net,
  CASE WHEN personal_retro3 IS NOT NULL THEN IF(personal_retro3 = 'Better', 1, 0) END   AS personal_better_ns_as_same,
  CASE WHEN personal_retro3 IS NOT NULL THEN IF(personal_retro3 = 'Worse',  1, 0) END   AS personal_worse_ns_as_same,
  personal_score5,

  -- ── Controls ───────────────────────────────────────────────────────────────
  age,
  CASE
    WHEN age BETWEEN 18 AND 34 THEN '18-34'
    WHEN age BETWEEN 35 AND 54 THEN '35-54'
    WHEN age >= 55             THEN '55+'
  END                                           AS age_group,         -- same cut points as UMich a1834 / a3554 / a5597
  sex,
  educ3,
  race4,
  income_tercile,
  employment5,
  owns_home,
  is_stock_investor,
  state_abbr,
  county_fips,
  zipcode,
  interview_mode
FROM with_white_house;

## 4. `int_econ_monthly` — the fundamentals, one row per month
**Grain:** one row per calendar month from January 2000 to the latest month any series reaches (October 2026 for daily
stock and weekly mortgage data; most monthly series end in August or September 2026).

### How it is built
1. **Month spine.** `explode(sequence(first, last, 1 month))` creates every month exactly once. All series are LEFT JOINed onto
   it, so a month with no data is a row of NULLs, never a missing row.
2. **Year-over-year and month-over-month changes use `LAG` over the spine.** Because the spine has no gaps, `LAG(x, 12)` is
   guaranteed to be the same month a year earlier (the handoff's rule: *lag by date, not by row*, is satisfied by construction).
3. **Higher-frequency series are collapsed to months:** weekly mortgage rate → monthly average; daily stock indices →
   month-end close (last trading day) plus the date of that close, so a partial month (October 2026) is visible.
4. **Quarterly series stay on their release month only** (credit card and auto loan rates in Feb/May/Aug/Nov; debt service,
   real weekly earnings at the start of each quarter). Other months are NULL. **Never forward-filled.**

### Unemployment
`unemployment_rate_derived_pct` = misery index − headline inflation, rounded to one decimal (BLS publishes one decimal). This
is a stand-in until official `UNRATE` is added (a listed gap). **October 2025:** the misery index file carries a value (7.29)
for a month when neither CPI nor the unemployment rate was published, so `misery_index_pct` is set to NULL for that month
here (staging keeps the published value and its flag) and derived unemployment is NULL too.

### Coverage to know about
* CPI components start January 2016, so **their year-over-year changes start January 2017.** Headline inflation (separate file)
  goes back to 2000. For a 2016 window, only headline inflation is available, so the RQ2 gas/grocery salience test starts in
  2017 unless CPI levels for 2015 are added (new data gap, free from BLS).
* Misery index starts June 2016; FRED rates and markets mostly start in late 2016; real PCE starts 2007 (YoY from 2008).
* `white_house_party` is the party in office on the 1st of the month, so January of an inauguration year belongs to the outgoing
  party. Event studies use the explicit event months in `int_event_panel` instead.

In [0]:
%sql
CREATE OR REPLACE TABLE int_econ_monthly
COMMENT 'Monthly fundamentals on a gap-free month spine (Jan 2000 onward): headline inflation, derived unemployment, misery index, CPI component YoY, real earnings, real PCE, saving rate, rates, month-end stock closes. Oct 2025 left NULL; quarterly series only on release months; nothing filled.'
AS
WITH
-- ─────────────────────────────────────────────────────────────────────────────
-- 1. Month spine: first month of inflation data through the last month any source covers.
-- ─────────────────────────────────────────────────────────────────────────────
bounds AS (
  SELECT
    (SELECT MIN(month) FROM stg_inflation)                     AS first_month,
    greatest(
      (SELECT MAX(month) FROM stg_inflation),
      (SELECT MAX(month) FROM stg_cpi),
      (SELECT MAX(month) FROM stg_misery_index),
      (SELECT MAX(trunc(observation_date, 'MM')) FROM stg_fred_observations)
    )                                                          AS last_month
),

spine AS (
  SELECT explode(sequence(first_month, last_month, INTERVAL 1 MONTH)) AS month
  FROM bounds
),

-- ─────────────────────────────────────────────────────────────────────────────
-- 2. FRED series collapsed to one row per month (long -> wide).
-- ─────────────────────────────────────────────────────────────────────────────
fred_monthly AS (
  SELECT
    trunc(observation_date, 'MM')                                          AS month,
    -- monthly series (one observation per month already)
    MAX(IF(series_id = 'PCEC96',  value, NULL))                            AS pce_real_bn_2017usd,
    MAX(IF(series_id = 'PSAVERT', value, NULL))                            AS personal_saving_rate_pct,
    -- weekly -> monthly average
    ROUND(AVG(IF(series_id = 'MORTGAGE30US', value, NULL)), 3)             AS mortgage_rate_30y_avg_pct,
    -- quarterly series: a value only on their release month, NULL otherwise (no fill)
    MAX(IF(series_id = 'TERMCBCCALLNS',   value, NULL))                    AS credit_card_rate_pct,
    MAX(IF(series_id = 'TERMCBAUTO48NS',  value, NULL))                    AS auto_loan_48m_rate_pct,
    MAX(IF(series_id = 'TDSP',            value, NULL))                    AS debt_service_pct_disposable_income,
    MAX(IF(series_id = 'LES1252881600Q',  value, NULL))                    AS median_weekly_real_earnings_1982usd
  FROM stg_fred_observations
  GROUP BY trunc(observation_date, 'MM')
),

-- Daily stock indices -> month-end close. MAX_BY(value, date) = the value on the latest trading day in the month.
stocks_monthly AS (
  SELECT
    trunc(observation_date, 'MM')                                          AS month,
    MAX_BY(IF(series_id = 'SP500',     value, NULL), IF(series_id = 'SP500',     observation_date, NULL)) AS sp500_close_eom,
    MAX_BY(IF(series_id = 'DJIA',      value, NULL), IF(series_id = 'DJIA',      observation_date, NULL)) AS djia_close_eom,
    MAX_BY(IF(series_id = 'NASDAQCOM', value, NULL), IF(series_id = 'NASDAQCOM', observation_date, NULL)) AS nasdaq_close_eom,
    MAX(IF(series_id = 'SP500', observation_date, NULL))                                                  AS stock_last_trading_date
  FROM stg_fred_observations
  WHERE series_id IN ('SP500', 'DJIA', 'NASDAQCOM')
  GROUP BY trunc(observation_date, 'MM')
),

-- ─────────────────────────────────────────────────────────────────────────────
-- 3. Everything on the spine (LEFT JOINs keep every month).
-- ─────────────────────────────────────────────────────────────────────────────
joined AS (
  SELECT
    s.month,
    i.inflation_yoy_pct,
    -- Drop the source-filled October 2025 misery value: no official inputs exist for that month.
    IF(m.is_filled_by_source, NULL, m.misery_index_pct)                    AS misery_index_pct,
    c.cpi_all_items,
    c.cpi_core,
    c.cpi_food_at_home,
    c.cpi_gasoline,
    c.cpi_shelter,
    c.real_avg_hourly_earnings_prod_nonsup,
    f.pce_real_bn_2017usd,
    f.personal_saving_rate_pct,
    f.mortgage_rate_30y_avg_pct,
    f.credit_card_rate_pct,
    f.auto_loan_48m_rate_pct,
    f.debt_service_pct_disposable_income,
    f.median_weekly_real_earnings_1982usd,
    k.sp500_close_eom,
    k.djia_close_eom,
    k.nasdaq_close_eom,
    k.stock_last_trading_date
  FROM spine AS s
  LEFT JOIN stg_inflation    AS i ON s.month = i.month
  LEFT JOIN stg_misery_index AS m ON s.month = m.month
  LEFT JOIN stg_cpi          AS c ON s.month = c.month
  LEFT JOIN fred_monthly     AS f ON s.month = f.month
  LEFT JOIN stocks_monthly   AS k ON s.month = k.month
),

-- ─────────────────────────────────────────────────────────────────────────────
-- 4. Changes. LAG over the gap-free spine: LAG(x, 12) is always 12 months earlier, LAG(x, 1) the month before.
--    Any NULL on either side (e.g. October 2025) gives a NULL change rather than a wrong one.
-- ─────────────────────────────────────────────────────────────────────────────
with_changes AS (
  SELECT
    *,
    ROUND(100 * (cpi_all_items    / LAG(cpi_all_items, 12)    OVER w - 1), 2) AS cpi_all_items_yoy_pct,
    ROUND(100 * (cpi_core         / LAG(cpi_core, 12)         OVER w - 1), 2) AS cpi_core_yoy_pct,
    ROUND(100 * (cpi_food_at_home / LAG(cpi_food_at_home, 12) OVER w - 1), 2) AS cpi_food_at_home_yoy_pct,
    ROUND(100 * (cpi_gasoline     / LAG(cpi_gasoline, 12)     OVER w - 1), 2) AS cpi_gasoline_yoy_pct,
    ROUND(100 * (cpi_shelter      / LAG(cpi_shelter, 12)      OVER w - 1), 2) AS cpi_shelter_yoy_pct,
    ROUND(100 * (real_avg_hourly_earnings_prod_nonsup
                 / LAG(real_avg_hourly_earnings_prod_nonsup, 12) OVER w - 1), 2) AS real_avg_hourly_earnings_yoy_pct,
    ROUND(100 * (pce_real_bn_2017usd / LAG(pce_real_bn_2017usd, 1)  OVER w - 1), 3) AS pce_real_mom_pct,
    ROUND(100 * (pce_real_bn_2017usd / LAG(pce_real_bn_2017usd, 12) OVER w - 1), 2) AS pce_real_yoy_pct,
    ROUND(100 * (sp500_close_eom     / LAG(sp500_close_eom, 1)      OVER w - 1), 2) AS sp500_mom_pct
  FROM joined
  WINDOW w AS (ORDER BY month)
)

SELECT
  -- ── Calendar ───────────────────────────────────────────────────────────────
  e.month,
  year(e.month)                                                AS year,
  t.party                                                      AS white_house_party,       -- in office on the 1st of the month
  e.month = DATE'2025-10-01'                                   AS is_shutdown_gap_month,   -- no CPI or unemployment published

  -- ── Headline benchmarks ────────────────────────────────────────────────────
  e.inflation_yoy_pct,                                                                     -- headline CPI, % vs 12 months earlier
  e.misery_index_pct,
  ROUND(e.misery_index_pct - e.inflation_yoy_pct, 1)           AS unemployment_rate_derived_pct,

  -- ── CPI components (year-over-year %, from SA index levels) ────────────────
  e.cpi_all_items_yoy_pct,                                                                 -- cross-check vs inflation_yoy_pct (QA)
  e.cpi_core_yoy_pct,
  e.cpi_food_at_home_yoy_pct,                                                              -- "visible" price: groceries
  e.cpi_gasoline_yoy_pct,                                                                  -- "visible" price: gas
  e.cpi_shelter_yoy_pct,

  -- ── Paychecks ──────────────────────────────────────────────────────────────
  e.real_avg_hourly_earnings_prod_nonsup,                                                  -- 1982-84 dollars, production/nonsupervisory
  e.real_avg_hourly_earnings_yoy_pct,
  e.median_weekly_real_earnings_1982usd,                                                   -- quarterly; release months only

  -- ── Household behavior (RQ4 "do" side) ─────────────────────────────────────
  e.pce_real_bn_2017usd,
  e.pce_real_mom_pct,
  e.pce_real_yoy_pct,
  e.personal_saving_rate_pct,
  e.debt_service_pct_disposable_income,                                                    -- quarterly; release months only

  -- ── Cost of credit ─────────────────────────────────────────────────────────
  e.mortgage_rate_30y_avg_pct,
  e.credit_card_rate_pct,                                                                  -- quarterly; release months only
  e.auto_loan_48m_rate_pct,                                                                -- quarterly; release months only

  -- ── Markets (price indices, not total return) ──────────────────────────────
  e.sp500_close_eom,
  e.sp500_mom_pct,
  e.djia_close_eom,
  e.nasdaq_close_eom,
  e.stock_last_trading_date                                                                -- < month end = partial month
FROM with_changes AS e
LEFT JOIN int_presidential_terms AS t
  ON e.month BETWEEN t.term_start_date AND t.term_end_date;

## 5. `int_sentiment_monthly` — what people say, one row per month
**Grain:** one row per month, January 2016 to the last UMich month (August 2026).

Puts the perception series side by side: full-sample sentiment indices, the three party indices (from April 2017), the partisan
gap, and price expectations next to **actual** inflation.

### Two "expected vs actual" measures (a decision for the panel)
UMich asks in month *m* what prices will do over the **next** 12 months. Two comparisons are possible:
| Column | Compares the month-*m* expectation with | Answers |
|---|---|---|
| `expected_minus_current_inflation_pct` | inflation over the **past** 12 months (YoY in month *m*) | "Do people expect prices to run hotter than what the news says they just did?" This is the plan's preliminary measure (−3.0 in 2022, +2.2 in 2025). |
| `forecast_error_12m_pct` | inflation over the **following** 12 months (YoY in month *m* + 12) | "Were people right?" A true forecast error; only defined through the latest month minus 12. |

They answer different questions, so both are kept and named for what they compare. The story should say which one it uses.

### Units
Sentiment indices: 1966 = 100. Price expectations: percent. **Gas expectations are cents per gallon**, not percent; converting
them needs the retail pump price in dollars, which is not in the repo (FRED `GASREGW` would fill this gap). They are carried here
unconverted and labeled.

In [0]:
%sql
CREATE OR REPLACE TABLE int_sentiment_monthly
COMMENT 'Monthly perception series Jan 2016 onward: UMich full-sample sentiment indices, party indices (Apr 2017+), Rep-Dem gap, year-ahead price expectations next to actual current and 12-months-ahead inflation.'
AS
WITH
bounds AS (
  SELECT MIN(month) AS first_month, MAX(month) AS last_month FROM stg_umich_index
),

spine AS (
  SELECT explode(sequence(first_month, last_month, INTERVAL 1 MONTH)) AS month FROM bounds
),

-- Long UMich file -> the handful of full-sample measures used downstream (subgroup 'all').
-- Subgroup series stay in stg_umich_index; marts query it directly when they need them.
umich_all AS (
  SELECT
    month,
    MAX(IF(measure_code = 'ics',         value, NULL)) AS ics_all,          -- Index of Consumer Sentiment
    MAX(IF(measure_code = 'icc',         value, NULL)) AS icc_all,          -- current conditions
    MAX(IF(measure_code = 'ice',         value, NULL)) AS ice_all,          -- expectations
    MAX(IF(measure_code = 'pago_r',      value, NULL)) AS personal_finances_vs_year_ago_score,  -- the UMich "wallet" item
    MAX(IF(measure_code = 'px1_med',     value, NULL)) AS expected_inflation_1y_median_pct,     -- RQ2 primary
    MAX(IF(measure_code = 'px1_mean',    value, NULL)) AS expected_inflation_1y_mean_pct,
    MAX(IF(measure_code = 'px5_med',     value, NULL)) AS expected_inflation_5y_median_pct,
    MAX(IF(measure_code = 'gas1px_md',   value, NULL)) AS expected_gas_change_1y_median_cents   -- cents per gallon, NOT percent
  FROM stg_umich_index
  WHERE subgroup_code = 'all'
  GROUP BY month
)

SELECT
  s.month,
  year(s.month)                                                AS year,
  e.white_house_party,

  -- ── Sentiment ──────────────────────────────────────────────────────────────
  u.ics_all,
  u.icc_all,
  u.ice_all,
  u.personal_finances_vs_year_ago_score,
  p.sentiment_rep,                                             -- NULL before April 2017
  p.sentiment_dem,
  p.sentiment_ind,
  ROUND(p.sentiment_rep - p.sentiment_dem, 1)                  AS partisan_gap_rep_minus_dem,
  -- Gap signed so positive = the White House party's supporters feel better than the other party's.
  ROUND(CASE e.white_house_party
          WHEN 'Republican' THEN p.sentiment_rep - p.sentiment_dem
          WHEN 'Democrat'   THEN p.sentiment_dem - p.sentiment_rep
        END, 1)                                                AS partisan_gap_in_minus_out,

  -- ── Price expectations vs actual inflation ─────────────────────────────────
  u.expected_inflation_1y_median_pct,
  u.expected_inflation_1y_mean_pct,
  u.expected_inflation_5y_median_pct,
  u.expected_gas_change_1y_median_cents,
  e.inflation_yoy_pct                                          AS actual_inflation_current_pct,      -- past 12 months
  e12.inflation_yoy_pct                                        AS actual_inflation_next_12m_pct,     -- following 12 months
  ROUND(u.expected_inflation_1y_median_pct - e.inflation_yoy_pct, 2)   AS expected_minus_current_inflation_pct,
  ROUND(u.expected_inflation_1y_mean_pct   - e.inflation_yoy_pct, 2)   AS expected_mean_minus_current_inflation_pct,
  ROUND(u.expected_inflation_1y_median_pct - e12.inflation_yoy_pct, 2) AS forecast_error_12m_pct
FROM spine AS s
LEFT JOIN umich_all        AS u   ON s.month = u.month
LEFT JOIN stg_umich_party  AS p   ON s.month = p.month
LEFT JOIN int_econ_monthly AS e   ON s.month = e.month
LEFT JOIN int_econ_monthly AS e12 ON e12.month = add_months(s.month, 12);   -- realized inflation one year later

## 6. `int_econ_windows` — the economy each survey's respondents had just lived through
**Grain:** one row per source × survey year × window length (1, 3 or 6 months).

Implements the plan's assumption **A5**: respondents react to conditions with a short lag, measured as a **trailing window
ending the month before fieldwork**. Three months is primary; 1 and 6 months are the pre-specified robustness windows (they are
all built up front so none is chosen by searching).

* **CES fieldwork month** = month of the median respondent start date that year (even years ≈ October, odd years ≈ November).
* **ANES fieldwork month** = October (the assumed reference date from `int_respondents`; flagged).
* Each window reports **how many months actually had data** (`n_months_*`) and `is_complete_window`. `AVG` skips NULLs, so an
  incomplete window is a shorter average, not a filled one.
* `*_chg_vs_prev_wave` = change from the same window in the previous wave of that survey (previous year for CES, four years for
  ANES). These are the "year-over-year changes in fundamentals" RQ2 correlates with changes in independents' ratings.

> **The 2025 CES window touches October 2025.** CES 2025 fielded in November–December, so its 3-month window is Aug–Oct 2025
> and its 1-month window is October 2025 alone, the shutdown month with no CPI or unemployment data. The 3-month value is a
> 2-month average and the 1-month value is NULL. 2025 is the flagship comparison year, so the write-up should name this, and the
> mart should either report the 2-month average as such or use the 6-month window for 2025. The QA cell shows the coverage.

In [0]:
%sql
CREATE OR REPLACE TABLE int_econ_windows
COMMENT 'Trailing economic windows (1, 3, 6 months ending the month before fieldwork) for each CES and ANES survey year: average fundamentals, months of data actually available, and change vs the previous wave.'
AS
WITH
-- Fieldwork month per survey and year.
fieldwork AS (
  SELECT
    source,
    survey_year,
    -- Median start date. percentile_disc needs a number, so dates go through unix_date (days since 1970) and back.
    date_from_unix_date(CAST(percentile_disc(0.5) WITHIN GROUP (ORDER BY unix_date(fieldwork_ref_date)) AS INT))
                                                       AS fieldwork_median_date,
    MIN(fieldwork_ref_date)                            AS fieldwork_first_date,
    MAX(fieldwork_ref_date)                            AS fieldwork_last_date,
    BOOL_OR(fieldwork_date_is_assumed)                 AS fieldwork_date_is_assumed,
    COUNT(*)                                           AS n_respondents
  FROM int_respondents
  GROUP BY source, survey_year
),

window_lengths AS (
  SELECT * FROM VALUES (1, FALSE), (3, TRUE), (6, FALSE) AS t(window_months, is_primary_window)
),

windows AS (
  SELECT
    f.*,
    w.window_months,
    w.is_primary_window,
    trunc(f.fieldwork_median_date, 'MM')                              AS fieldwork_month,
    add_months(trunc(f.fieldwork_median_date, 'MM'), -w.window_months) AS window_start_month,
    add_months(trunc(f.fieldwork_median_date, 'MM'), -1)              AS window_end_month     -- month BEFORE fieldwork
  FROM fieldwork AS f
  CROSS JOIN window_lengths AS w
),

averaged AS (
  SELECT
    w.source,
    w.survey_year,
    w.window_months,
    w.is_primary_window,
    w.fieldwork_median_date,
    w.fieldwork_first_date,
    w.fieldwork_last_date,
    w.fieldwork_date_is_assumed,
    w.n_respondents,
    w.fieldwork_month,
    w.window_start_month,
    w.window_end_month,

    -- Data coverage inside the window (NULL months are skipped by AVG, so these say what each average is made of)
    COUNT(e.inflation_yoy_pct)                         AS n_months_inflation,
    COUNT(e.unemployment_rate_derived_pct)             AS n_months_unemployment,
    COUNT(e.cpi_gasoline_yoy_pct)                      AS n_months_cpi_components,

    -- Window averages
    ROUND(AVG(e.inflation_yoy_pct), 2)                 AS inflation_yoy_pct,
    ROUND(AVG(e.unemployment_rate_derived_pct), 2)     AS unemployment_rate_pct,
    ROUND(AVG(e.misery_index_pct), 2)                  AS misery_index_pct,
    ROUND(AVG(e.cpi_core_yoy_pct), 2)                  AS cpi_core_yoy_pct,
    ROUND(AVG(e.cpi_gasoline_yoy_pct), 2)              AS cpi_gasoline_yoy_pct,
    ROUND(AVG(e.cpi_food_at_home_yoy_pct), 2)          AS cpi_food_at_home_yoy_pct,
    ROUND(AVG(e.cpi_shelter_yoy_pct), 2)               AS cpi_shelter_yoy_pct,
    ROUND(AVG(e.real_avg_hourly_earnings_yoy_pct), 2)  AS real_avg_hourly_earnings_yoy_pct,
    ROUND(AVG(e.pce_real_yoy_pct), 2)                  AS pce_real_yoy_pct,
    ROUND(AVG(e.mortgage_rate_30y_avg_pct), 2)         AS mortgage_rate_30y_pct
  FROM windows AS w
  LEFT JOIN int_econ_monthly AS e
    ON e.month BETWEEN w.window_start_month AND w.window_end_month
  GROUP BY ALL
)

SELECT
  *,
  -- Complete = every month in the window has headline inflation and unemployment.
  n_months_inflation = window_months AND n_months_unemployment = window_months      AS is_complete_window,

  -- Change vs the same window in the previous wave of the same survey (CES: previous year; ANES: four years earlier).
  LAG(survey_year)                       OVER prev                                 AS prev_wave_year,
  ROUND(inflation_yoy_pct        - LAG(inflation_yoy_pct)        OVER prev, 2)     AS inflation_chg_vs_prev_wave,
  ROUND(unemployment_rate_pct    - LAG(unemployment_rate_pct)    OVER prev, 2)     AS unemployment_chg_vs_prev_wave,
  ROUND(misery_index_pct         - LAG(misery_index_pct)         OVER prev, 2)     AS misery_chg_vs_prev_wave,
  ROUND(cpi_gasoline_yoy_pct     - LAG(cpi_gasoline_yoy_pct)     OVER prev, 2)     AS gasoline_yoy_chg_vs_prev_wave,
  ROUND(cpi_food_at_home_yoy_pct - LAG(cpi_food_at_home_yoy_pct) OVER prev, 2)     AS food_at_home_yoy_chg_vs_prev_wave,
  ROUND(real_avg_hourly_earnings_yoy_pct
        - LAG(real_avg_hourly_earnings_yoy_pct) OVER prev, 2)                      AS real_earnings_yoy_chg_vs_prev_wave
FROM averaged
WINDOW prev AS (PARTITION BY source, window_months ORDER BY survey_year);

## 7 · `int_event_panel` — event time around every election
**Grain:** one row per event × month (−6 to +14 months from the election month) × series. Series are the three UMich party
indices plus the full-sample index (`'All respondents'`), which reaches back to January 2016 and so covers the 2016–17
transition that the party series misses.

**Events:** all 7 presidential elections in `ref_events` (5 party changes + the 2004 and 2012 re-elections as placebos) and the
5 midterms in `ref_midterms`. Rows exist for every event even where sentiment is unavailable, because the economic columns
(inflation, unemployment) feed the plan's fundamentals-at-transitions table for every transition, including 2016–17.

### Columns that encode the plan's design
* `months_from_election`, `months_from_inauguration`: the two co-primary event clocks (plan revision of Oct 22). For a November
  election the inauguration month is election + 2. Midterms have no inauguration (NULL).
* `party_role`: **Winning party / Losing party / Independent / All respondents.** For presidential elections the winner is the
  incoming president's party; for midterms it is the party that took control of the flipped chamber. This makes the symmetric
  framing explicit: every chart compares "the side that won" with "the side that lost", whichever party that was.
* `baseline_index`: mean of months −4 to −2 (July–September before a November election), only when all three months exist.
  `change_from_baseline` = index − baseline.

Economic columns repeat across the four series rows of the same month: filter to one series before summing or averaging them.
The "share of swing complete" estimand and its bootstrap interval are computed in the mart / Python layer from this table; the
QA cell below reproduces the plan's preliminary version of that table.

In [0]:
%sql
CREATE OR REPLACE TABLE int_event_panel
COMMENT 'Event-study panel: presidential elections 2000-2024 and midterms 2006-2022, months -6 to +14 from the election, UMich sentiment by party (and full sample) with July-September baseline, plus inflation and unemployment for the fundamentals table.'
AS
WITH
events AS (
  -- Presidential elections (transitions and re-elections)
  SELECT
    concat('pres-', CAST(election_year AS STRING))     AS event_key,
    'Presidential election'                            AS event_type,
    transition_label                                   AS event_label,
    election_date,
    election_month,
    inauguration_month,
    incoming_party                                     AS winning_party,
    outgoing_party                                     AS white_house_party_at_election,
    party_changed                                      AS white_house_changed_parties,
    CAST(NULL AS STRING)                               AS chambers_flipped
  FROM ref_events

  UNION ALL

  -- Midterms (placebos: Congress changes hands, White House does not)
  SELECT
    concat('mid-', CAST(m.election_year AS STRING)),
    'Midterm election',
    concat(CAST(m.election_year AS STRING), ' midterm'),
    m.election_date,
    m.election_month,
    CAST(NULL AS DATE),
    m.winning_party,
    t.party,                                           -- White House party on Election Day
    FALSE,
    m.chambers_flipped
  FROM ref_midterms AS m
  LEFT JOIN int_presidential_terms AS t
    ON m.election_date BETWEEN t.term_start_date AND t.term_end_date
),

event_months AS (
  SELECT explode(sequence(-6, 14)) AS months_from_election
),

-- Sentiment long: one row per month x series. stack() turns the four columns into rows.
sentiment_long AS (
  SELECT month, series, sentiment_index
  FROM int_sentiment_monthly
  LATERAL VIEW stack(4,
      'Republican',      sentiment_rep,
      'Democrat',        sentiment_dem,
      'Independent',     sentiment_ind,
      'All respondents', ics_all
  ) st AS series, sentiment_index
),

series_list AS (
  SELECT * FROM VALUES ('Republican'), ('Democrat'), ('Independent'), ('All respondents') AS t(series)
),

-- Full grid: every event x every event month x every series (sentiment may be NULL outside coverage).
grid AS (
  SELECT
    ev.*,
    em.months_from_election,
    add_months(ev.election_month, em.months_from_election)                         AS month,
    -- Inauguration clock: election clock minus the election-to-inauguration gap (2 months for November elections).
    em.months_from_election
      - CAST(months_between(ev.inauguration_month, ev.election_month) AS INT)      AS months_from_inauguration,
    sl.series
  FROM events AS ev
  CROSS JOIN event_months AS em
  CROSS JOIN series_list  AS sl
),

with_values AS (
  SELECT
    g.*,
    CASE
      WHEN g.series IN ('Independent', 'All respondents') THEN g.series
      WHEN g.series = g.winning_party                     THEN 'Winning party'
      ELSE 'Losing party'
    END                                                                             AS party_role,
    s.sentiment_index,
    e.white_house_party                                                             AS white_house_party_in_month,
    e.inflation_yoy_pct,
    e.unemployment_rate_derived_pct,
    e.misery_index_pct,
    e.cpi_gasoline_yoy_pct,
    e.real_avg_hourly_earnings_yoy_pct,
    e.pce_real_yoy_pct
  FROM grid AS g
  LEFT JOIN sentiment_long   AS s ON g.month = s.month AND g.series = s.series
  LEFT JOIN int_econ_monthly AS e ON g.month = e.month
),

with_baseline AS (
  SELECT
    *,
    -- Baseline: months -4..-2 (Jul-Sep before a November election), per event and series.
    AVG(CASE WHEN months_from_election BETWEEN -4 AND -2 THEN sentiment_index END)
      OVER (PARTITION BY event_key, series)                                         AS baseline_raw,
    COUNT(CASE WHEN months_from_election BETWEEN -4 AND -2 THEN sentiment_index END)
      OVER (PARTITION BY event_key, series)                                         AS baseline_n_months
  FROM with_values
)

SELECT
  event_key,
  event_type,
  event_label,
  election_date,
  election_month,
  inauguration_month,
  winning_party,
  white_house_party_at_election,
  white_house_changed_parties,
  chambers_flipped,
  month,
  months_from_election,
  months_from_inauguration,
  series,
  party_role,
  sentiment_index,
  IF(baseline_n_months = 3, ROUND(baseline_raw, 2), NULL)                           AS baseline_index,  -- only with all 3 months
  IF(baseline_n_months = 3, ROUND(sentiment_index - baseline_raw, 2), NULL)         AS change_from_baseline,
  white_house_party_in_month,
  inflation_yoy_pct,
  unemployment_rate_derived_pct,
  misery_index_pct,
  cpi_gasoline_yoy_pct,
  real_avg_hourly_earnings_yoy_pct,
  pce_real_yoy_pct
FROM with_baseline;

## 8. `int_pew_polls` — Pew toplines positioned around each transition
**Grain:** one row per poll × series (Total, Republican, Democrat): 28 polls × 3 = 84 rows.

The plan's Pew design is *last poll before vs first poll after each inauguration*, read as a difference in differences of party
shares. This model does the bookkeeping:
* Each poll is assigned to the **nearest party-changing transition** (by distance to the inauguration date) and gets signed day
  counts to that election and inauguration.
* `poll_period` = Before election / Between election and inauguration / After inauguration.
* `is_last_before_inauguration` and `is_first_after_inauguration` mark the two polls that bracket each inauguration.
* `partisan_role` uses the White House party **on the poll date**.

> **The 2016-11-08 poll is dated Election Day.** Whether `Date` is fieldwork end or publication is unverified (handoff item 11),
> so this poll may be pre- or post-election. It is flagged `is_dated_on_election_day`; resolve against Pew's methodology page
> before using it as the 2016 "before" poll.

In [0]:
%sql
CREATE OR REPLACE TABLE int_pew_polls
COMMENT 'Pew economy ratings, one row per poll x series (Total/Republican/Democrat), assigned to the nearest party-changing transition with days from election and inauguration, poll period, bracketing-poll flags and partisan role.'
AS
WITH
transitions AS (
  SELECT transition_label, election_date, inauguration_date, incoming_party, outgoing_party
  FROM ref_events
  WHERE party_changed
),

-- Nearest transition per poll (MIN_BY picks the transition with the smallest distance to its inauguration).
nearest AS (
  SELECT
    p.poll_date,
    MIN_BY(t.transition_label,  abs(datediff(p.poll_date, t.inauguration_date))) AS transition_label,
    MIN_BY(t.election_date,     abs(datediff(p.poll_date, t.inauguration_date))) AS election_date,
    MIN_BY(t.inauguration_date, abs(datediff(p.poll_date, t.inauguration_date))) AS inauguration_date
  FROM stg_pew AS p
  CROSS JOIN transitions AS t
  GROUP BY p.poll_date
),

polls AS (
  SELECT
    p.*,
    n.transition_label,
    n.election_date,
    n.inauguration_date,
    datediff(p.poll_date, n.election_date)                       AS days_from_election,
    datediff(p.poll_date, n.inauguration_date)                   AS days_from_inauguration,
    CASE
      WHEN p.poll_date <  n.election_date     THEN 'Before election'
      WHEN p.poll_date <  n.inauguration_date THEN 'Between election and inauguration'
      ELSE 'After inauguration'
    END                                                          AS poll_period,
    p.poll_date = n.election_date                                AS is_dated_on_election_day,
    t.party                                                      AS white_house_party
  FROM workspace.public_opinion_economy_case_study.stg_pew AS p
  JOIN nearest AS n ON p.poll_date = n.poll_date
  LEFT JOIN int_presidential_terms AS t
    ON p.poll_date BETWEEN t.term_start_date AND t.term_end_date
),

flagged AS (
  SELECT
    *,
    -- Last poll before / first poll after the inauguration, within each transition.
    poll_date = MAX(IF(poll_date <  inauguration_date, poll_date, NULL)) OVER (PARTITION BY transition_label) AS is_last_before_inauguration,
    poll_date = MIN(IF(poll_date >= inauguration_date, poll_date, NULL)) OVER (PARTITION BY transition_label) AS is_first_after_inauguration
  FROM polls
)

SELECT
  f.poll_date,
  f.transition_label,
  f.election_date,
  f.inauguration_date,
  f.days_from_election,
  f.days_from_inauguration,
  f.poll_period,
  f.is_dated_on_election_day,
  COALESCE(f.is_last_before_inauguration, FALSE)               AS is_last_before_inauguration,
  COALESCE(f.is_first_after_inauguration, FALSE)               AS is_first_after_inauguration,
  f.white_house_party,
  st.series,
  CASE
    WHEN st.series = 'Total'                THEN 'All respondents'
    WHEN st.series = f.white_house_party    THEN 'In-party'
    ELSE 'Out-party'
  END                                                          AS partisan_role,
  st.pct_excellent_good
FROM flagged AS f
LATERAL VIEW stack(3,
    'Total',      pct_excellent_good_total,
    'Republican', pct_excellent_good_rep,
    'Democrat',   pct_excellent_good_dem
) st AS series, pct_excellent_good;

## 9. QA checks
Each check either returns 0 failures or reproduces a number already reported in the plan, so a wrong join or a recoding error
shows up as a mismatch. In dbt these become `schema.yml` tests (grain, accepted values) and singular tests (reconciliations).

**9a · Grain and integrity.** Every `failures` value should be **0**.

In [0]:
%sql
SELECT 'int_respondents: duplicate respondent_key'                AS check, COUNT(*) - COUNT(DISTINCT respondent_key) AS failures FROM int_respondents
UNION ALL SELECT 'int_respondents: no White House party',                   COUNT(*) FROM int_respondents WHERE white_house_party IS NULL
UNION ALL SELECT 'int_respondents: rows outside 2016+ window',              COUNT(*) FROM int_respondents WHERE survey_year < 2016
UNION ALL SELECT 'int_respondents: econ_better set but retro3 is Not sure', COUNT(*) FROM int_respondents WHERE econ_retro3 = 'Not sure' AND econ_better IS NOT NULL
UNION ALL SELECT 'int_respondents: CES row with personal finances',         COUNT(*) FROM int_respondents WHERE source = 'CES' AND personal_retro3 IS NOT NULL
UNION ALL SELECT 'int_respondents: Independent with co_partisan <> 0',      COUNT(*) FROM int_respondents WHERE party3 = 'Independent' AND co_partisan <> 0
UNION ALL SELECT 'int_presidential_terms: overlapping terms',               COUNT(*) FROM int_presidential_terms a JOIN int_presidential_terms b
                                                                              ON a.term_start_date < b.term_start_date AND a.term_end_date >= b.term_start_date
UNION ALL SELECT 'int_econ_monthly: duplicate month',                       COUNT(*) - COUNT(DISTINCT month) FROM int_econ_monthly
UNION ALL SELECT 'int_econ_monthly: gaps in month spine',
          CAST(months_between(MAX(month), MIN(month)) + 1 - COUNT(*) AS BIGINT) FROM int_econ_monthly
UNION ALL SELECT 'int_econ_monthly: Oct 2025 has a filled value',           COUNT(*) FROM int_econ_monthly
          WHERE is_shutdown_gap_month AND (inflation_yoy_pct IS NOT NULL OR misery_index_pct IS NOT NULL OR unemployment_rate_derived_pct IS NOT NULL)
UNION ALL SELECT 'int_sentiment_monthly: duplicate month',                  COUNT(*) - COUNT(DISTINCT month) FROM int_sentiment_monthly
UNION ALL SELECT 'int_econ_windows: duplicate source/year/window',          COUNT(*) - COUNT(DISTINCT source, survey_year, window_months) FROM int_econ_windows
UNION ALL SELECT 'int_event_panel: duplicate event/month/series',           COUNT(*) - COUNT(DISTINCT event_key, month, series) FROM int_event_panel
UNION ALL SELECT 'int_pew_polls: rows (expect 84) minus 84',                COUNT(*) - 84 FROM int_pew_polls;

**9b. Row counts and coverage at a glance.**

In [0]:
%sql
SELECT 'ref_midterms' AS model, COUNT(*) AS n_rows, CAST(MIN(election_date) AS STRING) AS first, CAST(MAX(election_date) AS STRING) AS last FROM ref_midterms
UNION ALL SELECT 'int_presidential_terms', COUNT(*), CAST(MIN(term_start_date) AS STRING), CAST(MAX(term_start_date) AS STRING) FROM int_presidential_terms
UNION ALL SELECT 'int_respondents (CES)',  COUNT(*), CAST(MIN(survey_year) AS STRING), CAST(MAX(survey_year) AS STRING) FROM int_respondents WHERE source = 'CES'
UNION ALL SELECT 'int_respondents (ANES)', COUNT(*), CAST(MIN(survey_year) AS STRING), CAST(MAX(survey_year) AS STRING) FROM int_respondents WHERE source = 'ANES'
UNION ALL SELECT 'int_econ_monthly',       COUNT(*), CAST(MIN(month) AS STRING), CAST(MAX(month) AS STRING) FROM int_econ_monthly
UNION ALL SELECT 'int_sentiment_monthly',  COUNT(*), CAST(MIN(month) AS STRING), CAST(MAX(month) AS STRING) FROM int_sentiment_monthly
UNION ALL SELECT 'int_econ_windows',       COUNT(*), CAST(MIN(survey_year) AS STRING), CAST(MAX(survey_year) AS STRING) FROM int_econ_windows
UNION ALL SELECT 'int_event_panel',        COUNT(*), CAST(MIN(month) AS STRING), CAST(MAX(month) AS STRING) FROM int_event_panel
UNION ALL SELECT 'int_pew_polls',          COUNT(*), CAST(MIN(poll_date) AS STRING), CAST(MAX(poll_date) AS STRING) FROM int_pew_polls;

**9c. CES headline, rebuilt from `int_respondents`.** Should match notebook 01's QA 12c:
main spec (not sure excluded) Dem 52.5 → 5.1, Rep 5.0 → 52.2, Ind 11.7 → 13.1; not sure folded into "same" (the plan's
preliminary numbers) Dem 50.8 → 5.1, Rep 4.9 → 50.8, Ind 10.9 → 12.2. The `partisan_role` column also shows the flip: the
in-party is optimistic in both years, whichever party it is.

In [0]:
%sql
SELECT
  survey_year,
  party3,
  partisan_role,
  ROUND(100 * SUM(econ_better * weight)            / SUM(IF(econ_better IS NOT NULL, weight, 0)), 1)            AS pct_better_main,
  ROUND(100 * SUM(econ_better_ns_as_same * weight) / SUM(IF(econ_better_ns_as_same IS NOT NULL, weight, 0)), 1) AS pct_better_ns_as_same,
  ROUND(100 * SUM(econ_net * weight)               / SUM(IF(econ_net IS NOT NULL, weight, 0)), 1)               AS net_rating_main,
  COUNT(*)                                                                                                       AS n
FROM int_respondents
WHERE source = 'CES' AND survey_year IN (2024, 2025) AND party3 IN ('Democrat', 'Republican', 'Independent')
GROUP BY survey_year, party3, partisan_role
ORDER BY party3, survey_year;